# Taxonomy Batch Labelling (Mistral)

Set de clustering (`mistral_extraction_full`, single-record, non-multitask) :

1. **Tâche** — assigner `paraphrase_task` à la taxonomie manuelle (mother/leaf).
2. **Domaine d'application** — classifier `title` + `abstract` vers le vocabulaire `THEMATIC_DOMAINS` (13 valeurs).

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path().resolve().parent))

In [ ]:
from src.taxonomy.providers import mistral_batch_results, mistral_batch_submit
from src.taxonomy.schemas import TASKS, THEMATIC_DOMAINS, VALID_TAXONOMY

## 1. Shared config

In [ ]:
MODEL = "mistral-large-latest"
RUN_NAME = "clustering_set"

DATA_DIR = Path("../data")
TAXONOMY_DIR = DATA_DIR / "taxonomy"
PER_LLM_DIR = TAXONOMY_DIR / "per_llm"
LEAF_OUT_DIR = TAXONOMY_DIR / "leaf_assignments"
DOMAIN_OUT_DIR = TAXONOMY_DIR / "domain_assignments"
LEAF_OUT_DIR.mkdir(parents=True, exist_ok=True)
DOMAIN_OUT_DIR.mkdir(parents=True, exist_ok=True)

LEAF_TASK = "taxonomy_leaf_assignment"
LEAF_OUT_PATH = LEAF_OUT_DIR / f"{RUN_NAME}_mistral_leaf_assignments.parquet"

DOMAIN_TASK = "thematic_domain_assignment"
DOMAIN_OUT_PATH = DOMAIN_OUT_DIR / f"{RUN_NAME}_mistral_thematic_domains.parquet"

leaf_cfg = TASKS[LEAF_TASK]
domain_cfg = TASKS[DOMAIN_TASK]
print("leaf:", leaf_cfg.tool_name, leaf_cfg.max_tokens)
print(
    "taxonomy:",
    len(VALID_TAXONOMY),
    "mothers,",
    sum(len(v) for v in VALID_TAXONOMY.values()),
    "leaves",
)
print(
    "domain:",
    domain_cfg.tool_name,
    domain_cfg.max_tokens,
    "|",
    len(THEMATIC_DOMAINS),
    "labels",
)

## 2. Load Clustering Set

In [ ]:
df = pd.read_parquet(PER_LLM_DIR / "mistral_extraction_full.parquet").merge(
    pd.read_parquet(PER_LLM_DIR / "google.parquet")[
        ["bibkey", "google_task_type", "title", "abstract"]
    ],
    on="bibkey",
)

print(f"After load+merge: {len(df):,}")

required = {"bibkey", "title", "abstract"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

df["is_multi_record"] = df.groupby("bibkey")["bibkey"].transform("size") > 1
df = df[~df.is_multi_record]
print(f"After multi-record filter: {len(df):,}")

df = df[(~df.is_multitask) & (~df.subtasks_not_inferable)]
print(f"After multitask filter: {len(df):,}")

df = df.dropna(subset=["bibkey", "title", "abstract"]).sort_values("bibkey")
df = df.reset_index(drop=True)

df[["bibkey", "title", "abstract", "paraphrase_task"]].head(3)

In [ ]:
SMOKE_TEST = False  # False → clustering set complet (1357)

# ~2 papiers par catégorie cible (proxy: domain d'extraction + relecture titre/abstract)
SMOKE_TEST_BY_CATEGORY: dict[str, list[str]] = {
    "general": ["pinter-etal-2020-nytwit", "gong-etal-2020-design"],
    "news": ["shen-etal-2024-multilingual", "agrawal-etal-2022-towards"],
    "biomedical": ["akhtar-etal-2022-pubhealthtab"],
    "mental_health": ["agarwal-etal-2025-redepress"],
    "legal": ["thalken-etal-2023-modeling", "bocchi-etal-2024-kevlar"],
    "scientific": ["casey-etal-2019-classifying"],
    "education": ["hawasly-etal-2025-arabicweb", "amorim-etal-2018-automated"],
    "finance": ["chen-etal-2018-bq", "conforti-etal-2022-incorporating"],
    "e_commerce": ["agrahari-etal-2025-really"],
    "literary": ["piper-budac-2025-cr4", "kabir-etal-2023-banglabook"],
    "politics": ["menini-tonelli-2016-agreement", "mohammad-etal-2016-dataset"],
    "linguistics": [
        "hamalainen-etal-2021-finnish",
        "sanchez-bayona-agerri-2022-leveraging",
    ],
    "other": ["cho-seo-2025-towards", "biamby-etal-2022-twitter"],
}

if SMOKE_TEST:
    smoke_bibkeys = [b for keys in SMOKE_TEST_BY_CATEGORY.values() for b in keys]
    missing = sorted(set(smoke_bibkeys) - set(df["bibkey"]))
    if missing:
        raise ValueError(f"Smoke test bibkeys absent du set: {missing}")

    category_by_bibkey = {
        b: cat for cat, keys in SMOKE_TEST_BY_CATEGORY.items() for b in keys
    }
    df = (
        df[df["bibkey"].isin(smoke_bibkeys)]
        .assign(_smoke_category=lambda d: d["bibkey"].map(category_by_bibkey))
        .sort_values(["_smoke_category", "bibkey"])
        .reset_index(drop=True)
    )
    print(
        f"[smoke test] {len(df):,} papers — {df['_smoke_category'].nunique()} catégories"
    )
    display(
        df.groupby("_smoke_category", as_index=False)
        .agg(n=("bibkey", "count"), example_title=("title", "first"))
        .sort_values("_smoke_category")
    )
else:
    print(f"{len(df):,} papers")

## 3. Leaf assignment — submit batch

Run once. Keep the printed `job_id`.

In [ ]:
items = [
    (
        str(row.bibkey),
        {"paper_text": row.paraphrase_task},
    )
    for row in df.itertuples(index=False)
]

leaf_job_id = mistral_batch_submit(items, model=MODEL, task=LEAF_TASK)
leaf_job_id

## 4. Leaf assignment — fetch results

À lancer quand le batch Mistral est terminé. Les assignations invalides `mother`/`leaf` sont filtrées pendant le fetch.

In [ ]:
LEAF_JOB_ID = "47d3de90-340c-499b-8dc9-78b83ff7c695"

assigned = mistral_batch_results(LEAF_JOB_ID, df, task=LEAF_TASK)
assigned.to_parquet(LEAF_OUT_PATH, index=False)
LEAF_OUT_PATH

## 5. Leaf assignment — quick inspection

In [ ]:
mother_col = "mistral_mother"
leaf_col = "mistral_leaf"

print("rows:", len(assigned))
missing_assignments = assigned[mother_col].isna().sum()
print("missing/invalid assignments:", missing_assignments)

display(assigned[mother_col].value_counts(dropna=False).head(30))
display(assigned[["bibkey", "title", mother_col, leaf_col]].head(20))

## 6. Thematic domain — submit batch

Input = `title` + `abstract` (pas `paraphrase_domain`). Run once ; garder `domain_job_id`.

In [ ]:
domain_items = [
    (
        str(row.bibkey),
        {"title": row.title, "abstract": row.abstract},
    )
    for row in df.itertuples(index=False)
]

domain_job_id = mistral_batch_submit(domain_items, model=MODEL, task=DOMAIN_TASK)
domain_job_id

## 7. Thematic domain — fetch results

À lancer quand le batch est terminé. Les libellés hors vocabulaire sont coercés vers `other` côté schéma.

In [ ]:
DOMAIN_JOB_ID = "52a848d7-c0af-4c67-8a04-a6305dec73e5"

domain_assigned = mistral_batch_results(DOMAIN_JOB_ID, df, task=DOMAIN_TASK)
domain_assigned.to_parquet(DOMAIN_OUT_PATH, index=False)
DOMAIN_OUT_PATH

## 8. Thematic domain — quick inspection

In [ ]:
domain_col = "mistral_thematic_domain"

print("rows:", len(domain_assigned))
print("missing assignments:", domain_assigned[domain_col].isna().sum())

counts = domain_assigned[domain_col].value_counts(dropna=False)
other_pct = 100 * counts.get("other", 0) / len(domain_assigned)
print(f"other: {counts.get('other', 0):,} ({other_pct:.1f}%) — surveiller si > ~5 %")

display(counts)
display(domain_assigned[["bibkey", "title", domain_col, "mistral_reasoning"]].head(20))